# 📊 ChartVQA – Vietnamese Chart Visual Question Answering

Hệ thống **Visual Question Answering** đa nhiệm cho biểu đồ tiếng Việt, thực hiện đồng thời hai tác vụ:
- **Phân loại biểu đồ** (Chart Classification): nhận diện loại biểu đồ từ ảnh đầu vào.
- **Sinh câu trả lời** (Generative VQA): tạo câu trả lời dạng văn bản cho câu hỏi về nội dung biểu đồ.

---

## 🧠 Kiến trúc

**Encoder:**
- **[PhoBERT-base](https://huggingface.co/vinai/phobert-base)** — mã hóa câu hỏi tiếng Việt, trả về chuỗi hidden states `(B, L, 768)`.
- **[ViT-Base/16](https://huggingface.co/google/vit-base-patch16-224-in21k)** — trích xuất đặc trưng patch từ ảnh biểu đồ, trả về chuỗi `(B, 197, 768)`. Token `[CLS]` (vị trí 0) được dùng để phân loại.

**Fusion:**
- **CoAttention** (scaled dot-product cross-attention): text làm **query**, image làm **key/value**. Đặc trưng loại biểu đồ (`type_emb`) được nối vào đầu chuỗi text trước khi fusion, giúp decoder nhận biết ngữ cảnh loại biểu đồ.

**Classifier head:** Linear(768→256) → ReLU → Dropout(0.1) → Linear(256→num_classes). Phân loại loại biểu đồ từ `[CLS]` token của ViT.

**Decoder (sinh câu trả lời auto-regressive):**

| Cấu hình | Decoder | Chi tiết |
|---|---|---|
| **A1** | LSTM | Hidden state khởi tạo từ mean pooling của fused features |
| **A2** | Transformer Decoder | 3 layers, 8 attention heads, causal mask |

---

## ⚙️ 1. Cài đặt & Kết nối

In [ ]:
!pip install -q datasets evaluate bert_score rouge-score nltk

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = "/content/drive/MyDrive/VQA_Chart_Project"
os.makedirs(f"{PROJECT_PATH}/checkpoints", exist_ok=True)
print(f"✅ Project path: {PROJECT_PATH}")

## 🖥️ 2. Kiểm tra GPU

In [ ]:
import sys, torch
print(f"Python : {sys.executable}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Không có GPU'}")

## 📦 3. Import thư viện

In [ ]:
import os, shutil, random, json
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from datasets import load_dataset
from evaluate import load as eval_load
from transformers import AutoTokenizer, ViTImageProcessor, AutoModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import nltk

## 🔧 4. Cấu hình (Config)

In [ ]:
class Config:
    SEED            = 42
    PROJECT_PATH    = "/content/drive/MyDrive/VQA_Chart_Project/v2"
    CHECKPOINT_DIR  = f"{PROJECT_PATH}/checkpoints"

    LAST_CHECKPOINT = {
        "A1": f"{PROJECT_PATH}/last_checkpoint_a1.pth",
        "A2": f"{PROJECT_PATH}/last_checkpoint_a2.pth",
    }
    BEST_MODEL = {
        "A1": f"{PROJECT_PATH}/best_model_a1.pth",
        "A2": f"{PROJECT_PATH}/best_model_a2.pth",
    }

    MODEL_NAME_TEXT = "vinai/phobert-base"
    MODEL_NAME_IMG  = "google/vit-base-patch16-224-in21k"

    BATCH_SIZE        = 16
    MAX_LENGTH        = 64
    LEARNING_RATE     = 2e-5
    LAMBDA_CLS        = 0.3
    LABEL_SMOOTHING   = 0.1
    NUM_EPOCHS_WARMUP = 2
    NUM_EPOCHS_JOINT  = 10

    DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ANS_VOCAB_SIZE = 0

cfg = Config()

## 💾 5. Lưu / Phục hồi Checkpoint

In [ ]:
def save_checkpoint(state, is_best, epoch, cfg, type="A1"):
    assert type in cfg.LAST_CHECKPOINT
    os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)

    epoch_path = os.path.join(cfg.CHECKPOINT_DIR, f"{type.lower()}_epoch_{epoch}.pth")
    torch.save(state, epoch_path)
    torch.save(state, cfg.LAST_CHECKPOINT[type])

    if is_best:
        best_state = {
            'epoch':      state['epoch'],
            'best_acc':   state['best_acc'],
            'val_acc':    state['val_acc'],
            'state_dict': {
                k: v.half() if v.is_floating_point() else v
                for k, v in state['state_dict'].items()
            }
        }
        torch.save(best_state, cfg.BEST_MODEL[type])
        size_mb = os.path.getsize(cfg.BEST_MODEL[type]) / 1024 / 1024
        print(f"🌟 [{type}] Epoch {epoch}: Best model ({state['val_acc']:.4f}) — {size_mb:.1f} MB")

    size_mb = os.path.getsize(epoch_path) / 1024 / 1024
    print(f"💾 [{type}] Checkpoint: {epoch_path} — {size_mb:.1f} MB")


def load_checkpoint(model, optimizer, cfg, type="A1"):
    assert type in cfg.LAST_CHECKPOINT, f"❌ type '{type}' không hợp lệ. Chọn: {list(cfg.LAST_CHECKPOINT.keys())}"
    path = cfg.LAST_CHECKPOINT[type]
    if os.path.exists(path):
        print(f"♻️  [{type}] Phục hồi từ: {path}")
        ckpt = torch.load(path, map_location=cfg.DEVICE)
        model.load_state_dict(ckpt['state_dict'])
        optimizer.load_state_dict(ckpt['optimizer'])
        return ckpt['epoch'], ckpt['best_acc'], ckpt.get('history', [])
    print(f"⚠️  [{type}] Không tìm thấy checkpoint, bắt đầu từ đầu.")
    return 0, 0.0, []


def load_best_model(model, cfg, type="A1"):
    path = cfg.BEST_MODEL[type]
    if not os.path.exists(path):
        print(f"⚠️ Không tìm thấy best model: {path}")
        return
    ckpt = torch.load(path, map_location=cfg.DEVICE)
    state_dict = {k: v.float() if v.is_floating_point() else v for k, v in ckpt['state_dict'].items()}
    model.load_state_dict(state_dict)
    print(f"✅ [{type}] Load best model — Val Acc: {ckpt['val_acc']:.4f}")

## 📥 6. Tải Dataset

In [ ]:
dataset = load_dataset("Zenng2812/vqa-vietnamese-charts")

print(dataset)
print(f"\n✅ Train      : {len(dataset['train'])} mẫu")
print(f"✅ Validation : {len(dataset['validation'])} mẫu")
print(f"✅ Test        : {len(dataset['test'])} mẫu")

sample = dataset['train'][0]
print(f"\n❓ Câu hỏi     : {sample['question']}")
print(f"💬 Câu trả lời : {sample['answer']}")
print(f"🏷️ Loại biểu đồ: {sample['chart_type']}")

## 📖 7. Answer Vocabulary

In [ ]:
class AnswerVocab:
    """Vocabulary câu trả lời, xây dựng từ tập Train."""

    SPECIAL_TOKENS = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}

    def __init__(self, dataset, max_size=1000):
        self.word2idx = dict(self.SPECIAL_TOKENS)
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        all_words = " ".join(dataset['train']['answer']).lower().split()
        for word, _ in Counter(all_words).most_common(max_size):
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word

    def encode(self, text):
        unk = self.word2idx["<UNK>"]
        tokens = [self.word2idx.get(w, unk) for w in text.lower().split()]
        return [self.word2idx["<SOS>"]] + tokens + [self.word2idx["<EOS>"]]

    def decode(self, ids):
        skip = {self.word2idx["<PAD>"], self.word2idx["<SOS>"], self.word2idx["<EOS>"]}
        return " ".join(self.idx2word[i] for i in ids if i not in skip)

    @property
    def size(self):
        return len(self.word2idx)


ans_vocab = AnswerVocab(dataset)
cfg.ANS_VOCAB_SIZE = ans_vocab.size
print(f"✅ Vocabulary size: {cfg.ANS_VOCAB_SIZE}")

## 🔄 8. Tiền xử lý & DataLoader

In [ ]:
tokenizer       = AutoTokenizer.from_pretrained(cfg.MODEL_NAME_TEXT)
image_processor = ViTImageProcessor.from_pretrained(cfg.MODEL_NAME_IMG)

unique_types = sorted(set(dataset['train']['chart_type']))
type2id      = {t: i for i, t in enumerate(unique_types)}
id2type      = {i: t for t, i in type2id.items()}
print(f"✅ Chart types ({len(unique_types)}): {unique_types}")


def preprocess_fn(examples):
    inputs = tokenizer(
        examples['question'],
        padding="max_length", truncation=True, max_length=cfg.MAX_LENGTH
    )
    pixel_values = image_processor(images=examples['image'], return_tensors="pt").pixel_values
    return {
        "input_ids":      inputs['input_ids'],
        "attention_mask": inputs['attention_mask'],
        "pixel_values":   pixel_values,
        "chart_labels":   [type2id[t] for t in examples['chart_type']],
        "answer_ids":     [ans_vocab.encode(a) for a in examples['answer']],
    }


def collate_fn(batch):
    answer_ids = torch.nn.utils.rnn.pad_sequence(
        [item['answer_ids'] for item in batch],
        batch_first=True,
        padding_value=ans_vocab.word2idx["<PAD>"]
    )
    return {
        'input_ids':      torch.stack([item['input_ids']      for item in batch]),
        'attention_mask': torch.stack([item['attention_mask'] for item in batch]),
        'pixel_values':   torch.stack([item['pixel_values']   for item in batch]),
        'chart_labels':   torch.stack([item['chart_labels']   for item in batch]),
        'answer_ids':     answer_ids,
    }


processed_ds = dataset.map(preprocess_fn, batched=True, remove_columns=dataset['train'].column_names)
processed_ds.set_format("torch")

train_loader = DataLoader(processed_ds['train'],      batch_size=cfg.BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(processed_ds['validation'], batch_size=cfg.BATCH_SIZE, collate_fn=collate_fn)
test_loader  = DataLoader(processed_ds['test'],       batch_size=cfg.BATCH_SIZE, collate_fn=collate_fn)
print(f"✅ DataLoader sẵn sàng | Train: {len(train_loader)} batch | Val: {len(val_loader)} batch")

## 🏗️ 9. Kiến trúc Model

`ChartVQAModel` gồm:
- **Encoder**: PhoBERT (text) + ViT (image)
- **CoAttention**: Fusion đặc trưng text–image
- **Classifier head**: Phân loại loại biểu đồ
- **Decoder**: LSTM (A1) hoặc Transformer (A2)

In [ ]:
class CoAttention(nn.Module):
    """Cross-Attention: text query trên image key/value."""

    def __init__(self, embed_dim):
        super().__init__()
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.norm   = nn.LayerNorm(embed_dim)

    def forward(self, img_feats, text_feats):
        q     = self.q_proj(text_feats)
        k     = self.k_proj(img_feats)
        v     = self.v_proj(img_feats)
        scale = q.size(-1) ** 0.5
        attn  = F.softmax(torch.matmul(q, k.transpose(-2, -1)) / scale, dim=-1)
        fused = torch.matmul(attn, v)
        return self.norm(fused + text_feats)


class ChartVQAModel(nn.Module):
    """
    Mô hình đa nhiệm VQA cho biểu đồ tiếng Việt.
    - decoder_type='LSTM'        → Cấu hình A1
    - decoder_type='Transformer' → Cấu hình A2
    """

    def __init__(self, cfg, num_chart_types, decoder_type="LSTM"):
        super().__init__()
        self.decoder_type  = decoder_type
        self.text_encoder  = AutoModel.from_pretrained(cfg.MODEL_NAME_TEXT)
        self.image_encoder = AutoModel.from_pretrained(cfg.MODEL_NAME_IMG)
        dim = self.text_encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, num_chart_types)
        )
        self.fusion   = CoAttention(dim)
        self.type_emb = nn.Embedding(num_chart_types, dim)
        self.ans_emb  = nn.Embedding(cfg.ANS_VOCAB_SIZE, dim)

        if decoder_type == "LSTM":
            self.decoder = nn.LSTM(dim, dim, batch_first=True)
        else:
            layer        = nn.TransformerDecoderLayer(d_model=dim, nhead=8, batch_first=True)
            self.decoder = nn.TransformerDecoder(layer, num_layers=3)

        self.fc_out = nn.Linear(dim, cfg.ANS_VOCAB_SIZE)

    @staticmethod
    def make_causal_mask(sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, input_ids, attention_mask, pixel_values, chart_labels=None, ans_input=None):
        text_out   = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        img_out    = self.image_encoder(pixel_values=pixel_values).last_hidden_state
        logits_cls = self.classifier(img_out[:, 0, :])

        type_ids      = chart_labels if chart_labels is not None else logits_cls.argmax(dim=-1)
        context_token = self.type_emb(type_ids).unsqueeze(1)
        fused_features = self.fusion(img_out, torch.cat([context_token, text_out], dim=1))

        if ans_input is not None:
            ans_emb = self.ans_emb(ans_input)
            if self.decoder_type == "LSTM":
                h0 = fused_features.mean(dim=1).unsqueeze(0)
                outputs, _ = self.decoder(ans_emb, (h0, torch.zeros_like(h0)))
            else:
                tgt_mask = self.make_causal_mask(ans_input.size(1), ans_input.device)
                outputs  = self.decoder(tgt=ans_emb, memory=fused_features, tgt_mask=tgt_mask)
            return logits_cls, self.fc_out(outputs)

        return logits_cls, fused_features


print("✅ ChartVQAModel đã định nghĩa.")

## ⚡ 10. Loss Functions & Optimizer

In [ ]:
criterion_cls = nn.CrossEntropyLoss()
criterion_vqa = nn.CrossEntropyLoss(
    ignore_index=ans_vocab.word2idx["<PAD>"],
    label_smoothing=cfg.LABEL_SMOOTHING
)

def make_optimizer(model):
    return optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE)

print("✅ Loss functions & optimizer factory sẵn sàng.")

## 🔥 11. Phase 1 – Warm-up Classifier

Đóng băng Decoder, chỉ huấn luyện Encoder + Classifier head. Hỗ trợ resume từ checkpoint.

In [ ]:
def train_phase1(model, optimizer, train_loader, val_loader, cfg):
    print("🔥 Phase 1: Warm-up Classifier...")
    p1_path = os.path.join(cfg.CHECKPOINT_DIR, "phase1_latest.pth")
    start_epoch = 0

    if os.path.exists(p1_path):
        ckpt = torch.load(p1_path)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch']
        print(f"♻️ Tiếp tục từ epoch {start_epoch + 1}")

    for p in model.decoder.parameters():
        p.requires_grad = False

    for epoch in range(start_epoch, cfg.NUM_EPOCHS_WARMUP):
        model.train()
        total_acc = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")
        for batch in loop:
            ids    = batch['input_ids'].to(cfg.DEVICE)
            pv     = batch['pixel_values'].to(cfg.DEVICE)
            labels = batch['chart_labels'].to(cfg.DEVICE)
            mask   = batch['attention_mask'].to(cfg.DEVICE) if 'attention_mask' in batch else None
            optimizer.zero_grad()
            logits, _ = model(ids, mask, pv)
            loss = criterion_cls(logits, labels)
            loss.backward()
            optimizer.step()
            total_acc += (logits.argmax(1) == labels).float().mean().item()
            loop.set_postfix(train_acc=f"{total_acc/(loop.n+1):.4f}")

        model.eval()
        val_acc = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
                ids    = batch['input_ids'].to(cfg.DEVICE)
                pv     = batch['pixel_values'].to(cfg.DEVICE)
                labels = batch['chart_labels'].to(cfg.DEVICE)
                logits, _ = model(ids, None, pv)
                val_acc += (logits.argmax(1) == labels).float().mean().item()

        final_val_acc = val_acc / len(val_loader)
        ckpt_data = {
            'epoch': epoch + 1,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_acc': final_val_acc
        }
        torch.save(ckpt_data, p1_path)
        torch.save(ckpt_data, os.path.join(cfg.CHECKPOINT_DIR, f"phase1_epoch_{epoch+1}.pth"))
        print(f"💾 Epoch {epoch+1} | Val Acc: {final_val_acc:.4f}")

    for p in model.decoder.parameters():
        p.requires_grad = True
    print("✅ Phase 1 hoàn tất.")

## 🔍 12. Hàm Inference & Metrics

Bao gồm: sinh câu trả lời auto-regressive, exact match, soft accuracy, BLEU, ROUGE-L, BERTScore.

In [ ]:
def generate_answer(model, image, question, cfg, max_len=20):
    """Sinh câu trả lời auto-regressive (A1 và A2)."""
    model.eval()
    with torch.no_grad():
        pv = image.to(cfg.DEVICE) if isinstance(image, torch.Tensor)              else image_processor(images=image, return_tensors="pt").pixel_values.to(cfg.DEVICE)

        if isinstance(question, torch.Tensor):
            input_ids = question.to(cfg.DEVICE)
            attn_mask = torch.ones_like(input_ids)
        else:
            enc       = tokenizer(question, return_tensors="pt", padding=True,
                                  truncation=True, max_length=cfg.MAX_LENGTH).to(cfg.DEVICE)
            input_ids = enc.input_ids
            attn_mask = enc.attention_mask

        _, fused = model(input_ids, attn_mask, pv)

        SOS, EOS = ans_vocab.word2idx["<SOS>"], ans_vocab.word2idx["<EOS>"]
        predicted = []

        if model.decoder_type == "LSTM":
            curr = torch.tensor([[SOS]], device=cfg.DEVICE)
            h    = fused.mean(dim=1).unsqueeze(0)
            c    = torch.zeros_like(h)
            for _ in range(max_len):
                out, (h, c) = model.decoder(model.ans_emb(curr), (h, c))
                token = model.fc_out(out).argmax(dim=-1).item()
                if token == EOS: break
                predicted.append(token)
                curr = torch.tensor([[token]], device=cfg.DEVICE)
        else:
            seq = torch.tensor([[SOS]], device=cfg.DEVICE)
            for _ in range(max_len):
                mask  = ChartVQAModel.make_causal_mask(seq.size(1), cfg.DEVICE)
                out   = model.decoder(tgt=model.ans_emb(seq), memory=fused, tgt_mask=mask)
                token = model.fc_out(out[:, -1:, :]).argmax(dim=-1).item()
                if token == EOS: break
                predicted.append(token)
                seq = torch.cat([seq, torch.tensor([[token]], device=cfg.DEVICE)], dim=1)

    return ans_vocab.decode(predicted)


def validate(model, val_loader, cfg):
    """Exact Match Accuracy trên tập validation."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="🔍 Validating"):
            ans_ids = batch['answer_ids'].to(cfg.DEVICE)
            for i in range(len(batch['input_ids'])):
                pred   = generate_answer(model, batch['pixel_values'][i].unsqueeze(0),
                                         batch['input_ids'][i].unsqueeze(0), cfg)
                target = ans_vocab.decode(ans_ids[i].cpu().tolist())
                correct += pred.strip().lower() == target.strip().lower()
                total   += 1
    return correct / total


def soft_accuracy(prediction, reference):
    if isinstance(prediction, list): prediction = " ".join(prediction)
    if isinstance(reference,  list): reference  = " ".join(reference)
    pred_words = set(prediction.lower().split())
    ref_words  = set(reference.lower().split())
    if not ref_words: return 0.0
    return len(pred_words & ref_words) / len(ref_words)


def avg_soft_accuracy(predictions, references):
    scores = [soft_accuracy(p, r) for p, r in zip(predictions, references)]
    return round(sum(scores) / len(scores), 4) if scores else 0.0


def compute_metrics(predictions, references):
    """Tính BLEU, ROUGE-L, BERTScore."""
    smoother = SmoothingFunction().method1
    bleu_scores = [
        sentence_bleu([ref.strip().split()], pred.strip().split(), smoothing_function=smoother)
        for pred, ref in zip(predictions, references)
    ]

    scorer_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    rouge_scores = [scorer_rouge.score(ref, pred)['rougeL'].fmeasure for pred, ref in zip(predictions, references)]

    _, _, F1 = bert_score(predictions, references, lang="vi", verbose=False)

    return {
        "bleu":       round(sum(bleu_scores)  / len(bleu_scores)  if bleu_scores  else 0.0, 4),
        "rouge_l":    round(sum(rouge_scores) / len(rouge_scores) if rouge_scores else 0.0, 4),
        "bert_score": round(F1.mean().item(), 4),
    }


def decode_predictions(logits_vqa, target_output, idx2ans):
    """Decode logits → string để tính metrics."""
    pred_ids = logits_vqa.argmax(-1)
    predictions, references = [], []
    for pred_seq, ref_seq in zip(pred_ids, target_output):
        predictions.append(" ".join(idx2ans.get(t, "") for t in pred_seq.tolist() if t not in (0, 1, 2)))
        references.append(" ".join(idx2ans.get(t, "") for t in ref_seq.tolist()  if t not in (0, 1, 2)))
    return predictions, references


def exact_match_accuracy(predictions, references):
    correct = sum(p.strip().lower() == r.strip().lower() for p, r in zip(predictions, references))
    return round(correct / len(predictions), 4) if predictions else 0.0


def save_history(history, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=4)
    print(f"📝 History saved: {path}")


print("✅ Tất cả hàm inference & metrics sẵn sàng.")

## 🚀 13. Phase 2 – Joint Training

Huấn luyện đồng thời Classifier và VQA Decoder. Dùng chung cho cả A1 và A2.

In [ ]:
def train_joint(model, optimizer, train_loader, val_loader, cfg, idx2ans, tag="A1"):
    """Huấn luyện đồng thời Classifier + VQA Decoder. Hỗ trợ resume."""
    print(f"🚀 Phase 2 – Joint Training [{tag}]...")
    start_epoch, best_acc, history = load_checkpoint(model, optimizer, cfg, type=tag)
    history_path = os.path.join(cfg.CHECKPOINT_DIR, f"history_{tag.lower()}.json")

    for epoch in range(start_epoch, cfg.NUM_EPOCHS_JOINT):
        model.train()
        total_loss, total_vqa_loss, total_cls_loss = 0.0, 0.0, 0.0

        loop = tqdm(train_loader, desc=f"[{tag}] Epoch {epoch+1}/{cfg.NUM_EPOCHS_JOINT}")
        for batch in loop:
            ids    = batch['input_ids'].to(cfg.DEVICE)
            pv     = batch['pixel_values'].to(cfg.DEVICE)
            mask   = batch['attention_mask'].to(cfg.DEVICE) if 'attention_mask' in batch else None
            labels = batch['chart_labels'].to(cfg.DEVICE)
            ans    = batch['answer_ids'].to(cfg.DEVICE)

            decoder_input = ans[:, :-1]
            target_output = ans[:, 1:]

            optimizer.zero_grad()
            logits_cls, logits_vqa = model(ids, mask, pv, labels, decoder_input)
            loss_cls   = criterion_cls(logits_cls, labels)
            loss_vqa   = criterion_vqa(logits_vqa.reshape(-1, cfg.ANS_VOCAB_SIZE), target_output.reshape(-1))
            loss_total = loss_vqa + cfg.LAMBDA_CLS * loss_cls
            loss_total.backward()
            optimizer.step()

            total_loss     += loss_total.item()
            total_vqa_loss += loss_vqa.item()
            total_cls_loss += loss_cls.item()
            loop.set_postfix(loss=f"{loss_total.item():.4f}", vqa=f"{loss_vqa.item():.4f}")

        avg_loss     = total_loss     / len(train_loader)
        avg_vqa_loss = total_vqa_loss / len(train_loader)
        avg_cls_loss = total_cls_loss / len(train_loader)

        model.eval()
        all_preds, all_refs = [], []
        cls_correct, cls_total = 0, 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"[{tag}] Validating..."):
                ids    = batch['input_ids'].to(cfg.DEVICE)
                pv     = batch['pixel_values'].to(cfg.DEVICE)
                mask   = batch['attention_mask'].to(cfg.DEVICE) if 'attention_mask' in batch else None
                labels = batch['chart_labels'].to(cfg.DEVICE)
                ans    = batch['answer_ids'].to(cfg.DEVICE)

                logits_cls, logits_vqa = model(ids, mask, pv, labels, ans[:, :-1])
                cls_correct += (logits_cls.argmax(1) == labels).sum().item()
                cls_total   += labels.size(0)
                preds, refs  = decode_predictions(logits_vqa, ans[:, 1:], idx2ans)
                all_preds.extend(preds)
                all_refs.extend(refs)

        cls_acc       = round(cls_correct / cls_total, 4) if cls_total else 0.0
        exact_acc     = exact_match_accuracy(all_preds, all_refs)
        soft_acc      = avg_soft_accuracy(all_preds, all_refs)
        metrics       = compute_metrics(all_preds, all_refs)
        current_score = (exact_acc * 0.15 + soft_acc * 0.20 +
                         metrics['bleu'] * 0.15 + metrics['rouge_l'] * 0.20 + metrics['bert_score'] * 0.30)

        epoch_log = {
            "epoch": epoch + 1,
            "train": {"loss_total": round(avg_loss, 4), "loss_vqa": round(avg_vqa_loss, 4), "loss_cls": round(avg_cls_loss, 4)},
            "val":   {"cls_accuracy": cls_acc, "exact_match": exact_acc, "soft_accuracy": soft_acc, **metrics},
        }
        history.append(epoch_log)
        save_history(history, history_path)

        print(
            f"📊 [{tag}] Epoch {epoch+1} | Loss={avg_loss:.4f} | CLS={cls_acc:.4f} | "
            f"Soft={soft_acc:.4f} | EM={exact_acc:.4f} | "
            f"BLEU={metrics['bleu']:.4f} | ROUGE-L={metrics['rouge_l']:.4f} | BERT={metrics['bert_score']:.4f}"
        )

        is_best = current_score > best_acc
        if is_best:
            best_acc = current_score
            print(f"🏆 [NEW BEST: {current_score:.4f}] EM:{exact_acc:.2f} | Soft:{soft_acc:.2f} | "
                  f"BLEU:{metrics['bleu']:.2f} | ROUGE:{metrics['rouge_l']:.2f} | BERT:{metrics['bert_score']:.2f}")

        save_checkpoint({
            'epoch': epoch + 1, 'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(), 'best_acc': best_acc,
            'val_acc': best_acc, 'history': history,
        }, is_best, epoch + 1, cfg, type=tag)

    print(f"✅ [{tag}] Hoàn tất. Best Score: {best_acc:.4f}")
    return history

## 📏 14. Đánh giá trên tập Test (BERTScore)

In [ ]:
bertscore_metric = eval_load("bertscore")

def evaluate_soft_metrics(model, test_loader, cfg):
    model.eval()
    predictions, references = [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="📏 Evaluating"):
            ans_ids = batch['answer_ids']
            for i in range(len(batch['input_ids'])):
                pred   = generate_answer(model, batch['pixel_values'][i].unsqueeze(0),
                                         batch['input_ids'][i].unsqueeze(0), cfg)
                target = ans_vocab.decode(ans_ids[i].tolist())
                predictions.append(pred)
                references.append(target)

    results = bertscore_metric.compute(
        predictions=predictions, references=references,
        lang="vi", model_type=cfg.MODEL_NAME_TEXT
    )
    avg_f1 = sum(results['f1']) / len(results['f1'])
    print(f"✅ BERTScore F1: {avg_f1:.4f}")
    return avg_f1

print("✅ evaluate_soft_metrics sẵn sàng.")

## ▶️ 15. Chạy Cấu hình A1 – LSTM Decoder

In [ ]:
model_a1     = ChartVQAModel(cfg, num_chart_types=len(unique_types), decoder_type="LSTM").to(cfg.DEVICE)
optimizer_a1 = make_optimizer(model_a1)

train_phase1(model_a1, optimizer_a1, train_loader, val_loader, cfg)
train_joint(model_a1, optimizer_a1, train_loader, val_loader, cfg, ans_vocab.idx2word, tag="A1")

print("\n✨ Cấu hình A1 (LSTM) hoàn tất!")

## ▶️ 16. Chạy Cấu hình A2 – Transformer Decoder

In [ ]:
model_a2     = ChartVQAModel(cfg, num_chart_types=len(unique_types), decoder_type="Transformer").to(cfg.DEVICE)
optimizer_a2 = make_optimizer(model_a2)

p1_path = os.path.join(cfg.CHECKPOINT_DIR, "phase1_latest.pth")
if os.path.exists(p1_path):
    print("♻️ Nạp trọng số Encoder/Classifier từ Phase 1 (A1)...")
    ckpt = torch.load(p1_path)
    model_a2.load_state_dict(ckpt['model_state'], strict=False)

train_joint(model_a2, optimizer_a2, train_loader, val_loader, cfg, ans_vocab.idx2word, tag="A2")

print("\n✨ Cấu hình A2 (Transformer) hoàn tất!")

## 🖼️ 17. Dự đoán & Trực quan hóa

In [ ]:
model_a1 = ChartVQAModel(cfg, num_chart_types=len(unique_types), decoder_type="LSTM").to(cfg.DEVICE)
model_a2 = ChartVQAModel(cfg, num_chart_types=len(unique_types), decoder_type="Transformer").to(cfg.DEVICE)

load_best_model(model_a1, cfg, "A1")
load_best_model(model_a2, cfg, "A2")

In [ ]:
def predict_and_show(model, index=None, split="test"):
    """Hiển thị biểu đồ cùng câu hỏi, ground truth và câu trả lời dự đoán."""
    if index is None:
        index = random.randint(0, len(dataset[split]) - 1)

    sample = dataset[split][index]
    pred   = generate_answer(model, sample['image'], sample['question'], cfg)

    plt.figure(figsize=(9, 6))
    plt.imshow(sample['image'])
    plt.title(
        f"Q: {sample['question']}\n"
        f"🤖 Pred: {pred}\n"
        f"✅ GT  : {sample['answer']}",
        fontsize=11, loc='left'
    )
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"❓ Câu hỏi  : {sample['question']}")
    print(f"✅ Đáp án GT: {sample['answer']}")
    print(f"🤖 Dự đoán : {pred}")


predict_and_show(model_a2, index=10)